In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import time

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)
L = mp_pose.PoseLandmark

# joint -> (a, b, c) landmarks forming the angle at b, target angle, tolerance
JOINTS = {
    "L_ELBOW": (L.LEFT_SHOULDER, L.LEFT_ELBOW, L.LEFT_WRIST, 170, 15),
    "R_ELBOW": (L.RIGHT_SHOULDER, L.RIGHT_ELBOW, L.RIGHT_WRIST, 170, 15),
    "L_KNEE":  (L.LEFT_HIP, L.LEFT_KNEE, L.LEFT_ANKLE, 170, 15),
    "R_KNEE":  (L.RIGHT_HIP, L.RIGHT_KNEE, L.RIGHT_ANKLE, 170, 15),
}

# skeleton segments to draw: (landmark1, landmark2, joint_key_for_color)
SEGMENTS = [
    (L.LEFT_SHOULDER, L.LEFT_ELBOW, "L_ELBOW"),
    (L.LEFT_ELBOW, L.LEFT_WRIST, "L_ELBOW"),
    (L.RIGHT_SHOULDER, L.RIGHT_ELBOW, "R_ELBOW"),
    (L.RIGHT_ELBOW, L.RIGHT_WRIST, "R_ELBOW"),
    (L.LEFT_HIP, L.LEFT_KNEE, "L_KNEE"),
    (L.LEFT_KNEE, L.LEFT_ANKLE, "L_KNEE"),
    (L.RIGHT_HIP, L.RIGHT_KNEE, "R_KNEE"),
    (L.RIGHT_KNEE, L.RIGHT_ANKLE, "R_KNEE"),
    (L.LEFT_SHOULDER, L.RIGHT_SHOULDER, None),
    (L.LEFT_HIP, L.RIGHT_HIP, None),
    (L.LEFT_SHOULDER, L.LEFT_HIP, None),
    (L.RIGHT_SHOULDER, L.RIGHT_HIP, None),
]


def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ang = np.degrees(
        np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    )
    ang = abs(ang)
    return 360 - ang if ang > 180 else ang


def get_xy(lm, idx, w, h):
    p = lm[idx.value]
    return [p.x * w, p.y * h]


cap = cv2.VideoCapture(0)
prev_t = time.time()

while cap.isOpened():
    ok, frame = cap.read()
    if not ok:
        break
    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)

    angles = {}
    status_ok = True

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark

        for name, (a_idx, b_idx, c_idx, target, tol) in JOINTS.items():
            a = get_xy(lm, a_idx, w, h)
            b = get_xy(lm, b_idx, w, h)
            c = get_xy(lm, c_idx, w, h)
            ang = calculate_angle(a, b, c)
            angles[name] = ang
            if abs(ang - target) > tol:
                status_ok = False

        for p1, p2, joint_key in SEGMENTS:
            pt1 = tuple(np.int32(get_xy(lm, p1, w, h)))
            pt2 = tuple(np.int32(get_xy(lm, p2, w, h)))
            if joint_key is None:
                color = (255, 255, 255)
            else:
                target, tol = JOINTS[joint_key][3], JOINTS[joint_key][4]
                color = (0, 255, 0) if abs(angles[joint_key] - target) <= tol else (0, 0, 255)
            cv2.line(frame, pt1, pt2, color, 3)
            cv2.circle(frame, pt1, 5, (0, 255, 255), -1)
            cv2.circle(frame, pt2, 5, (0, 255, 255), -1)

        y = 90
        for name, ang in angles.items():
            cv2.putText(frame, f"{name}: {int(ang)}", (10, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            y += 25

        tag = "FORM MATCH" if status_ok else "INCORRECT FORM"
        tag_color = (0, 255, 0) if status_ok else (0, 0, 255)
        cv2.putText(frame, tag, (10, y + 10), cv2.FONT_HERSHEY_SIMPLEX,
                    0.9, tag_color, 3)

    curr_t = time.time()
    fps = 1 / (curr_t - prev_t) if curr_t != prev_t else 0
    prev_t = curr_t
    cv2.putText(frame, f"FPS: {int(fps)}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cv2.imshow("Workout Form Evaluator", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

AttributeError: partially initialized module 'cv2' has no attribute 'gapi_wip_gst_GStreamerPipeline' (most likely due to a circular import)